# Running Correlation Rainfall vs Nino3.4 di Target Box

Notebook ini menghitung running correlation DJF untuk rainfall saja.

Catatan:
- cache korelasi yang ada menyimpan korelasi statis full-period, jadi tidak bisa langsung dipakai untuk running correlation
- running correlation dihitung ulang dari data rainfall bulanan dan indeks Nino3.4 bulanan
- box 1: 95-125E, 6S-2N
- box 2: 100-115E, 5S-3N
- window: 9, 15, 17, dan 21 tahun, semuanya centered


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

plt.rcParams.update({
    'font.family': 'Helvetica',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
})

FULL_YEARS = np.arange(1981, 2026)
ANALYSIS_START = '1980-12-01'
ANALYSIS_END = '2025-02-28'
WINDOWS = [9, 15, 17, 21]

TARGET_BOX = {
    'lon_min': 95.0,
    'lon_max': 125.0,
    'lat_min': -6.0,
    'lat_max': 2.0,
}

TARGET_BOX_2 = {
    'lon_min': 100.0,
    'lon_max': 115.0,
    'lat_min': -5.0,
    'lat_max': 3.0,
}

RESULTS_DIR = Path('../../results/analisis_korelasi_26-19/running_correlation_targetbox').resolve()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RAINFALL_PATH = Path('/Users/rizzie/ClimateData/mswep-monthly/mswep_monthly_combined.nc').resolve()
NINO34_PATH = Path('/Users/rizzie/ClimateData/index-monthly/nino34.anom.csv').resolve()
NINO34_COLUMN = '   Nino Anom 3.4 Index  using ersstv5 from CPC  missing value -99.99 https://psl.noaa.gov/data/timeseries/month/'


In [ ]:
def standardize_obj(obj):
    rename_map = {}
    if 'valid_time' in obj.coords or 'valid_time' in obj.dims:
        rename_map['valid_time'] = 'time'
    if 'latitude' in obj.coords or 'latitude' in obj.dims:
        rename_map['latitude'] = 'lat'
    if 'longitude' in obj.coords or 'longitude' in obj.dims:
        rename_map['longitude'] = 'lon'
    if rename_map:
        obj = obj.rename(rename_map)

    if 'lon' in obj.coords:
        obj = obj.assign_coords(lon=(obj.lon % 360)).sortby('lon')

    if 'lat' in obj.coords:
        lat_values = np.asarray(obj['lat'].values)
        if lat_values.size > 1 and lat_values[0] < lat_values[-1]:
            obj = obj.sortby('lat', ascending=False)

    return obj


def select_target_box(da, box):
    lon_slice = slice(box['lon_min'], box['lon_max'])
    lat_values = np.asarray(da['lat'].values)
    if lat_values[0] > lat_values[-1]:
        lat_slice = slice(box['lat_max'], box['lat_min'])
    else:
        lat_slice = slice(box['lat_min'], box['lat_max'])
    return da.sel(lon=lon_slice, lat=lat_slice)


def build_djf_yearly_box_mean(da, box, name='rainfall'):
    da = standardize_obj(da)
    da = da.sel(time=slice(ANALYSIS_START, ANALYSIS_END))
    da = select_target_box(da, box)
    da = da.sel(time=da.time.dt.month.isin((12, 1, 2)))
    djf_year = xr.where(da.time.dt.month == 12, da.time.dt.year + 1, da.time.dt.year)
    da = da.assign_coords(djf_year=('time', djf_year.data))
    da = da.sel(time=da.djf_year.isin(FULL_YEARS))
    return da.mean(dim=('lat', 'lon')).groupby('djf_year').mean('time').rename(name)


def build_nino34_djf_series():
    df = pd.read_csv(
        NINO34_PATH,
        usecols=['Date', NINO34_COLUMN],
        parse_dates=['Date'],
    )
    df = df.set_index('Date').loc[ANALYSIS_START:ANALYSIS_END]
    df = df[df.index.month.isin((12, 1, 2))].copy()
    df['djf_year'] = df.index.year + (df.index.month == 12).astype('int8')
    df = df[df['djf_year'].isin(FULL_YEARS)].copy()
    series = df.groupby('djf_year')[NINO34_COLUMN].mean()
    out = pd.Series(series.to_numpy(), index=series.index.to_numpy(), name='nino34', dtype='float64')
    out.index.name = 'djf_year'
    return out


def to_pandas_series(da):
    out = pd.Series(da.values, index=da['djf_year'].values, name=da.name, dtype='float64')
    out.index.name = 'djf_year'
    return out.dropna()


def centered_running_corr(series, reference, window):
    pair = pd.concat([series, reference], axis=1, join='inner').dropna()
    pair.columns = ['rainfall', 'nino34']
    return pair['rainfall'].rolling(window=window, center=True, min_periods=window).corr(pair['nino34'])


def save_plot(fig, filename):
    fig.savefig(RESULTS_DIR / filename, dpi=300, bbox_inches='tight')


In [ ]:
rain_ds = xr.open_dataset(RAINFALL_PATH)
rainfall = build_djf_yearly_box_mean(rain_ds['precipitation'], TARGET_BOX, name='rainfall')
rainfall_series = to_pandas_series(rainfall)
nino34_djf = build_nino34_djf_series()

summary_df = pd.concat([rainfall_series, nino34_djf], axis=1, join='inner').dropna()
summary_df.head()


In [ ]:
running_corr = {
    window: centered_running_corr(rainfall_series, nino34_djf, window)
    for window in WINDOWS
}

pd.DataFrame(running_corr)


In [ ]:
line_colors = {
    9: '#1b9e77',
    15: '#d95f02',
    17: '#7570b3',
    21: '#e7298a',
}

fig, ax = plt.subplots(figsize=(11, 5.5))

for window in WINDOWS:
    ax.plot(
        running_corr[window].index,
        running_corr[window].values,
        linewidth=2.2,
        color=line_colors[window],
        label=f'{window}-year centered',
    )

ax.axhline(0.0, color='0.35', linewidth=1.0, linestyle='--')
ax.set_ylim(-1.0, 1.0)
ax.set_xlim(FULL_YEARS.min(), FULL_YEARS.max())
ax.set_xticks(np.arange(1981, 2026, 4))
ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.4)
ax.set_xlabel('DJF year', fontsize=12)
ax.set_ylabel('Running correlation rainfall vs Nino3.4', fontsize=12)
ax.set_title(
    'Rainfall: running correlation di target box 1\n'
    f"{TARGET_BOX['lon_min']:.0f}-{TARGET_BOX['lon_max']:.0f}E, {TARGET_BOX['lat_min']:.0f} sampai {TARGET_BOX['lat_max']:.0f} deg",
    fontsize=13,
    fontweight='bold',
    loc='left',
)
ax.legend(ncol=4, frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.16))

fig.tight_layout()
save_plot(fig, 'rainfall_running_correlation_target_box.png')
plt.show()
plt.close(fig)


In [ ]:
rainfall_2 = build_djf_yearly_box_mean(rain_ds['precipitation'], TARGET_BOX_2, name='rainfall')
rainfall_2_series = to_pandas_series(rainfall_2)

running_corr_box2 = {
    window: centered_running_corr(rainfall_2_series, nino34_djf, window)
    for window in WINDOWS
}

fig, ax = plt.subplots(figsize=(11, 5.5))
for window in WINDOWS:
    ax.plot(
        running_corr_box2[window].index,
        running_corr_box2[window].values,
        linewidth=2.2,
        color=line_colors[window],
        label=f'{window}-year centered',
    )

ax.axhline(0.0, color='0.35', linewidth=1.0, linestyle='--')
ax.set_ylim(-1.0, 1.0)
ax.set_xlim(FULL_YEARS.min(), FULL_YEARS.max())
ax.set_xticks(np.arange(1981, 2026, 4))
ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.4)
ax.set_xlabel('DJF year', fontsize=12)
ax.set_ylabel('Running correlation rainfall vs Nino3.4', fontsize=12)
ax.set_title(
    'Rainfall: running correlation di target box 2\n'
    f"{TARGET_BOX_2['lon_min']:.0f}-{TARGET_BOX_2['lon_max']:.0f}E, {TARGET_BOX_2['lat_min']:.0f} sampai {TARGET_BOX_2['lat_max']:.0f} deg",
    fontsize=13,
    fontweight='bold',
    loc='left',
)
ax.legend(ncol=4, frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.16))

fig.tight_layout()
save_plot(fig, 'rainfall_running_correlation_target_box2.png')
plt.show()
plt.close(fig)


In [ ]:
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())

ax.set_extent([92, 128, -8, 5], crs=ccrs.PlateCarree())
ax.coastlines(resolution='10m', linewidth=0.7, color='black')
ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.5, color='black')
ax.gridlines(draw_labels=False, linewidth=0.4, color='gray', alpha=0.4, linestyle='--')
ax.set_xticks(np.arange(95, 126, 5), crs=ccrs.PlateCarree())
ax.set_yticks(np.arange(-8, 6, 2), crs=ccrs.PlateCarree())
ax.xaxis.set_major_formatter(LongitudeFormatter())
ax.yaxis.set_major_formatter(LatitudeFormatter())

rect1 = plt.Rectangle(
    (TARGET_BOX['lon_min'], TARGET_BOX['lat_min']),
    TARGET_BOX['lon_max'] - TARGET_BOX['lon_min'],
    TARGET_BOX['lat_max'] - TARGET_BOX['lat_min'],
    facecolor='none',
    edgecolor='#1f77b4',
    linewidth=2.5,
    transform=ccrs.PlateCarree(),
    label='Target box 1',
)
rect2 = plt.Rectangle(
    (TARGET_BOX_2['lon_min'], TARGET_BOX_2['lat_min']),
    TARGET_BOX_2['lon_max'] - TARGET_BOX_2['lon_min'],
    TARGET_BOX_2['lat_max'] - TARGET_BOX_2['lat_min'],
    facecolor='none',
    edgecolor='#d62728',
    linewidth=2.5,
    transform=ccrs.PlateCarree(),
    label='Target box 2',
)

ax.add_patch(rect1)
ax.add_patch(rect2)
ax.legend(frameon=True, loc='lower left')
ax.set_title('Peta target box 1 dan 2', fontsize=13, fontweight='bold', loc='left')
fig.tight_layout()
save_plot(fig, 'target_boxes_map.png')
plt.show()
plt.close(fig)
